In [16]:
# CommerceIQ
# Phase 2: Data Acquisition and Ingestion
# Step 2.1: Define the expected dataset registry

DATASET_REGISTRY = {
    "orders": {
        "source": "ecommerce",
        "filename": "olist_orders_dataset.csv"
    },
    "customers": {
        "source": "ecommerce",
        "filename": "olist_customers_dataset.csv"
    },
    "order_items": {
        "source": "ecommerce",
        "filename": "olist_order_items_dataset.csv"
    },
    "payments": {
        "source": "ecommerce",
        "filename": "olist_order_payments_dataset.csv"
    },
    "reviews": {
        "source": "ecommerce",
        "filename": "olist_order_reviews_dataset.csv"
    },
    "products": {
        "source": "ecommerce",
        "filename": "olist_products_dataset.csv"
    },
    "sellers": {
        "source": "ecommerce",
        "filename": "olist_sellers_dataset.csv"
    },
    "geolocation": {
        "source": "ecommerce",
        "filename": "olist_geolocation_dataset.csv"
    },
    "category_translation": {
        "source": "ecommerce",
        "filename": "product_category_name_translation.csv"
    },
    "marketing_leads": {
        "source": "marketing",
        "filename": "olist_marketing_qualified_leads_dataset.csv"
    },
    "closed_deals": {
        "source": "marketing",
        "filename": "olist_closed_deals_dataset.csv"
    }
}

print("=" * 90)
print("COMMERCEIQ - EXPECTED DATASET REGISTRY")
print("=" * 90)

for name, details in DATASET_REGISTRY.items():
    print(
        f"{name:<23} | "
        f"{details['source']:<12} | "
        f"{details['filename']}"
    )

print("-" * 90)

print("Total expected datasets:", len(DATASET_REGISTRY))

print(
    "E-commerce datasets:",
    sum(
        details["source"] == "ecommerce"
        for details in DATASET_REGISTRY.values()
    )
)

print(
    "Marketing datasets:",
    sum(
        details["source"] == "marketing"
        for details in DATASET_REGISTRY.values()
    )
)

COMMERCEIQ - EXPECTED DATASET REGISTRY
orders                  | ecommerce    | olist_orders_dataset.csv
customers               | ecommerce    | olist_customers_dataset.csv
order_items             | ecommerce    | olist_order_items_dataset.csv
payments                | ecommerce    | olist_order_payments_dataset.csv
reviews                 | ecommerce    | olist_order_reviews_dataset.csv
products                | ecommerce    | olist_products_dataset.csv
sellers                 | ecommerce    | olist_sellers_dataset.csv
geolocation             | ecommerce    | olist_geolocation_dataset.csv
category_translation    | ecommerce    | product_category_name_translation.csv
marketing_leads         | marketing    | olist_marketing_qualified_leads_dataset.csv
closed_deals            | marketing    | olist_closed_deals_dataset.csv
------------------------------------------------------------------------------------------
Total expected datasets: 11
E-commerce datasets: 9
Marketing datasets: 2


In [17]:


# Step 2.2: Locate project root and load configuration

from pathlib import Path
import yaml

# Identify the current working directory
CURRENT_DIR = Path.cwd()

# Search the current directory and its parent directories
PROJECT_ROOT = next(
    (
        directory
        for directory in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (directory / "config.yaml").is_file()
        and (directory / "data" / "raw").is_dir()
    ),
    None
)

# Stop execution if the project root cannot be located
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "CommerceIQ project root could not be located."
    )

# Define the configuration file path
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

# Read and parse the YAML configuration
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)

# Display project configuration
print("=" * 80)
print("COMMERCEIQ - PROJECT CONFIGURATION")
print("=" * 80)

print("Current working directory:", CURRENT_DIR)
print("Project root directory:", PROJECT_ROOT)
print("Configuration file:", CONFIG_PATH)
print("Configuration data type:", type(CONFIG).__name__)

print("\nPROJECT DETAILS")

for key, value in CONFIG["project"].items():
    print(f"{key:<20}: {value}")

print("\nCONFIGURED DATA PATHS")

for key, value in CONFIG["paths"].items():
    print(f"{key:<20}: {value}")

print("\nDATABASE CONFIGURATION")
print("Database name:", CONFIG["database"]["name"])

print("\nLogging level:", CONFIG["logging"]["level"])

print("\nConfiguration loaded successfully.")

COMMERCEIQ - PROJECT CONFIGURATION
Current working directory: c:\Users\bgunj\Desktop\commerceIQ\notebooks
Project root directory: c:\Users\bgunj\Desktop\commerceIQ
Configuration file: c:\Users\bgunj\Desktop\commerceIQ\config.yaml
Configuration data type: dict

PROJECT DETAILS
name                : CommerceIQ
version             : 0.1.0
currency            : BRL

CONFIGURED DATA PATHS
raw_ecommerce       : data/raw/ecommerce
raw_marketing       : data/raw/marketing
raw_external        : data/raw/external
interim             : data/interim
processed           : data/processed
rejected            : data/rejected
metadata            : data/metadata
logs                : logs
reports             : reports

DATABASE CONFIGURATION
Database name: commerceiq

Logging level: INFO

Configuration loaded successfully.


In [18]:
# Step 2.3: Programmatically locate source files

SOURCE_PATHS = {
    "ecommerce": PROJECT_ROOT / CONFIG["paths"]["raw_ecommerce"],
    "marketing": PROJECT_ROOT / CONFIG["paths"]["raw_marketing"]
}

DATASET_PATHS = {}

for name, details in DATASET_REGISTRY.items():

    file_path = SOURCE_PATHS[details["source"]] / details["filename"]

    DATASET_PATHS[name] = file_path

print("COMMERCEIQ - SOURCE FILE DISCOVERY")
print("-" * 90)

for name, path in DATASET_PATHS.items():
    print(f"{name:<23} | {path.relative_to(PROJECT_ROOT)}")

print("-" * 90)
print("Total dataset paths generated:", len(DATASET_PATHS))

COMMERCEIQ - SOURCE FILE DISCOVERY
------------------------------------------------------------------------------------------
orders                  | data\raw\ecommerce\olist_orders_dataset.csv
customers               | data\raw\ecommerce\olist_customers_dataset.csv
order_items             | data\raw\ecommerce\olist_order_items_dataset.csv
payments                | data\raw\ecommerce\olist_order_payments_dataset.csv
reviews                 | data\raw\ecommerce\olist_order_reviews_dataset.csv
products                | data\raw\ecommerce\olist_products_dataset.csv
sellers                 | data\raw\ecommerce\olist_sellers_dataset.csv
geolocation             | data\raw\ecommerce\olist_geolocation_dataset.csv
category_translation    | data\raw\ecommerce\product_category_name_translation.csv
marketing_leads         | data\raw\marketing\olist_marketing_qualified_leads_dataset.csv
closed_deals            | data\raw\marketing\olist_closed_deals_dataset.csv
-----------------------------------

In [19]:
# Step 2.4: Validate source files

import pandas as pd

validation_results = []

for name, path in DATASET_PATHS.items():

    file_exists = path.is_file()
    correct_extension = path.suffix.lower() == ".csv"
    non_empty = file_exists and path.stat().st_size > 0

    status = (
        "PASSED"
        if file_exists and correct_extension and non_empty
        else "FAILED"
    )

    validation_results.append({
        "dataset": name,
        "file_exists": file_exists,
        "correct_extension": correct_extension,
        "non_empty": non_empty,
        "status": status
    })

validation_df = pd.DataFrame(validation_results)

print("COMMERCEIQ - SOURCE FILE VALIDATION")

display(validation_df)

passed = validation_df["status"].eq("PASSED").sum()
failed = validation_df["status"].eq("FAILED").sum()

print(f"Total datasets : {len(validation_df)}")
print(f"Passed         : {passed}")
print(f"Failed         : {failed}")

if failed > 0:
    raise ValueError("Source file validation failed.")

print("All required source files passed initial validation.")

COMMERCEIQ - SOURCE FILE VALIDATION


,dataset,file_exists,correct_extension,non_empty,status
0,orders,True,True,True,PASSED
1,customers,True,True,True,PASSED
2,order_items,True,True,True,PASSED
3,payments,True,True,True,PASSED
4,reviews,True,True,True,PASSED
5,products,True,True,True,PASSED
6,sellers,True,True,True,PASSED
7,geolocation,True,True,True,PASSED
8,category_translation,True,True,True,PASSED
9,marketing_leads,True,True,True,PASSED


Total datasets : 11
Passed         : 11
Failed         : 0
All required source files passed initial validation.


In [20]:
# Step 2.5: Capture source file metadata

from datetime import datetime, timezone

metadata_records = []

for name, path in DATASET_PATHS.items():

    file_stats = path.stat()

    metadata_records.append({
        "dataset": name,
        "source": DATASET_REGISTRY[name]["source"],
        "filename": path.name,
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": file_stats.st_size,
        "size_mb": round(file_stats.st_size / (1024 ** 2), 4),
        "last_modified_utc": datetime.fromtimestamp(
            file_stats.st_mtime,
            tz=timezone.utc
        ).isoformat()
    })

metadata_df = pd.DataFrame(metadata_records)

print("COMMERCEIQ - SOURCE FILE METADATA")

display(metadata_df[
    ["dataset", "source", "size_mb", "last_modified_utc"]
])

print(f"Total files     : {len(metadata_df)}")
print(f"Total size (MB) : {metadata_df['size_bytes'].sum() / (1024 ** 2):.2f}")
print("Source file metadata captured successfully.")

COMMERCEIQ - SOURCE FILE METADATA


,dataset,source,size_mb,last_modified_utc
0,orders,ecommerce,16.8370,2026-11-26T18:39:23.995135+00:00
1,customers,ecommerce,8.6155,2026-11-26T18:41:48.109973+00:00
2,order_items,ecommerce,14.7235,2026-11-26T18:39:40.961732+00:00
3,payments,ecommerce,5.5095,2026-11-26T18:39:35.476539+00:00
4,reviews,ecommerce,13.7822,2026-11-26T18:39:30.044304+00:00
5,products,ecommerce,2.2692,2026-11-26T18:38:35.571788+00:00
6,sellers,ecommerce,0.1666,2026-11-26T18:37:12.242203+00:00
7,geolocation,ecommerce,58.4353,2026-11-26T18:39:46.338390+00:00
8,category_translation,ecommerce,0.0025,2026-11-26T18:37:18.036313+00:00
9,marketing_leads,marketing,0.6712,2026-11-26T18:37:51.742755+00:00


Total files     : 11
Total size (MB) : 121.18
Source file metadata captured successfully.


In [21]:
# Step 2.6: Generate SHA-256 file hashes

import hashlib


def calculate_file_hash(file_path, chunk_size=1024 * 1024):
    """Generate a SHA-256 checksum for a file."""

    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


file_hashes = {}

for name, path in DATASET_PATHS.items():
    file_hashes[name] = calculate_file_hash(path)

# Add hashes to the existing metadata table
metadata_df["sha256"] = metadata_df["dataset"].map(file_hashes)

print("COMMERCEIQ - SOURCE FILE INTEGRITY")

display(metadata_df[["dataset", "sha256"]])

# Validate checksum format
valid_hashes = metadata_df["sha256"].str.fullmatch(
    r"[0-9a-f]{64}"
).fillna(False)

print(f"Total files    : {len(metadata_df)}")
print(f"Valid hashes   : {valid_hashes.sum()}")
print(f"Invalid hashes : {(~valid_hashes).sum()}")

if not valid_hashes.all():
    raise ValueError("SHA-256 checksum validation failed.")

print("SHA-256 checksums generated successfully.")

COMMERCEIQ - SOURCE FILE INTEGRITY


,dataset,sha256
0,orders,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c9...
1,customers,983a422239e1712ded753b3bf9ecf47dc73f144d306029...
2,order_items,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75f...
3,payments,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa9...
4,reviews,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307...
5,products,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b...
6,sellers,1f643d2b950373b85735e7794b20986f528d7a000432e7...
7,geolocation,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e0...
8,category_translation,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3...
9,marketing_leads,5537753b7549ce4068e52e2402c29f2773c5375fc05755...


Total files    : 11
Valid hashes   : 11
Invalid hashes : 0
SHA-256 checksums generated successfully.


In [22]:
# Step 2.7: Create and persist the dataset manifest

SOURCE_URLS = {
    "ecommerce": "https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce",
    "marketing": "https://www.kaggle.com/datasets/olistbr/marketing-funnel-olist"
}

# Combine metadata and validation results
manifest_df = metadata_df.merge(
    validation_df,
    on="dataset",
    how="left",
    validate="one_to_one"
)

# Add source provenance and registration timestamp
manifest_df["source_url"] = manifest_df["source"].map(SOURCE_URLS)

manifest_df["registered_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# Define output location
MANIFEST_PATH = (
    PROJECT_ROOT
    / CONFIG["paths"]["metadata"]
    / "dataset_manifest.csv"
)

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save the manifest
manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8"
)

# Read the saved manifest for verification
saved_manifest = pd.read_csv(MANIFEST_PATH)

assert len(saved_manifest) == len(DATASET_REGISTRY)
assert saved_manifest["dataset"].is_unique
assert saved_manifest["sha256"].equals(manifest_df["sha256"])
assert saved_manifest["source_url"].notna().all()
assert saved_manifest["status"].eq("PASSED").all()

print("COMMERCEIQ - DATASET MANIFEST")

display(saved_manifest[
    ["dataset", "source", "size_mb", "status"]
])

print(f"Manifest records : {len(saved_manifest)}")
print(f"Manifest columns : {len(saved_manifest.columns)}")
print(f"Total size (MiB) : {saved_manifest['size_bytes'].sum() / (1024 ** 2):.2f}")
print(f"Saved to         : {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"File exists      : {MANIFEST_PATH.is_file()}")
print("Manifest saved and verified successfully.")

COMMERCEIQ - DATASET MANIFEST


,dataset,source,size_mb,status
0,orders,ecommerce,16.8370,PASSED
1,customers,ecommerce,8.6155,PASSED
2,order_items,ecommerce,14.7235,PASSED
3,payments,ecommerce,5.5095,PASSED
4,reviews,ecommerce,13.7822,PASSED
5,products,ecommerce,2.2692,PASSED
6,sellers,ecommerce,0.1666,PASSED
7,geolocation,ecommerce,58.4353,PASSED
8,category_translation,ecommerce,0.0025,PASSED
9,marketing_leads,marketing,0.6712,PASSED


Manifest records : 11
Manifest columns : 14
Total size (MiB) : 121.18
Saved to         : data\metadata\dataset_manifest.csv
File exists      : True
Manifest saved and verified successfully.


In [23]:
# Step 2.8: Implement reusable CSV ingestion

from time import perf_counter


def load_csv_dataset(file_path):
    """Load a source CSV file without applying data cleaning."""

    return pd.read_csv(
        file_path,
        encoding="utf-8",
        low_memory=False,
        dtype=str,
        keep_default_na=False
    )


datasets = {}
ingestion_records = []

total_start = perf_counter()

for name, path in DATASET_PATHS.items():

    start_time = perf_counter()

    df = load_csv_dataset(path)

    datasets[name] = df

    ingestion_records.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mib": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        ),
        "load_time_sec": round(
            perf_counter() - start_time,
            4
        ),
        "status": "LOADED" if not df.empty else "EMPTY"
    })

ingestion_df = pd.DataFrame(ingestion_records)

print("COMMERCEIQ - CSV INGESTION SUMMARY")

display(ingestion_df)

print(f"Datasets loaded    : {len(datasets)}")
print(f"Total records      : {ingestion_df['rows'].sum():,}")
print(f"Total memory (MiB) : {ingestion_df['memory_mib'].sum():.2f}")
print(f"Total load time    : {perf_counter() - total_start:.2f} seconds")

assert len(datasets) == len(DATASET_REGISTRY)
assert ingestion_df["status"].eq("LOADED").all()

print("All source datasets loaded successfully.")

COMMERCEIQ - CSV INGESTION SUMMARY


,dataset,rows,columns,memory_mib,load_time_sec,status
0,orders,99441,8,53.02,0.4424,LOADED
1,customers,99441,5,30.95,0.1995,LOADED
2,order_items,112650,7,50.41,0.3108,LOADED
3,payments,103886,5,29.16,0.1614,LOADED
4,reviews,99224,7,45.46,0.3211,LOADED
5,products,32951,9,15.81,0.0728,LOADED
6,sellers,3095,4,0.72,0.0062,LOADED
7,geolocation,1000163,5,286.49,1.9641,LOADED
8,category_translation,71,2,0.01,0.0020,LOADED
9,marketing_leads,8000,4,2.14,0.0159,LOADED


Datasets loaded    : 11
Total records      : 1,559,764
Total memory (MiB) : 514.87
Total load time    : 3.51 seconds
All source datasets loaded successfully.


In [2]:
# Step 2.9(a): Test historical holiday API availability

import sys
import requests

print("Python interpreter:", sys.executable)

url = "https://date.nager.at/api/v3/PublicHolidays/2016/BR"

response = requests.get(url, timeout=30)

print("API URL:", response.url)
print("HTTP status:", response.status_code)

if response.status_code == 200:
    holiday_data = response.json()

    print("Records received:", len(holiday_data))
    print("Sample record:", holiday_data[0] if holiday_data else None)
else:
    print("API error:", response.text[:500])

Python interpreter: c:\Users\bgunj\Desktop\commerceIQ\.venv\Scripts\python.exe
API URL: https://date.nager.at/api/v3/PublicHolidays/2016/BR
HTTP status: 200
Records received: 14
Sample record: {'date': '2016-01-01', 'localName': 'Confraternização Universal', 'name': "New Year's Day", 'countryCode': 'BR', 'fixed': False, 'global': True, 'counties': None, 'launchYear': None, 'types': ['Public']}


In [1]:
import sys

print("Python interpreter:", sys.executable)
print("Environment prefix:", sys.prefix)
print("Base prefix:", sys.base_prefix)
print("Virtual environment active:", sys.prefix != sys.base_prefix)

Python interpreter: c:\Users\bgunj\Desktop\commerceIQ\.venv\Scripts\python.exe
Environment prefix: c:\Users\bgunj\Desktop\commerceIQ\.venv
Base prefix: C:\Program Files\Python313
Virtual environment active: True


In [5]:
from pathlib import Path
import yaml

CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        directory
        for directory in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (directory / "config.yaml").is_file()
        and (directory / "data" / "raw").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found.")

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)

print("Project root:", PROJECT_ROOT)
print("Configuration loaded:", isinstance(CONFIG, dict))
print("Holiday records available:", len(holidays_df))
print("External data path:", CONFIG["paths"]["raw_external"])

Project root: c:\Users\bgunj\Desktop\commerceIQ
Configuration loaded: True
Holiday records available: 42
External data path: data/raw/external


In [6]:
# Step 2.9(b): Save and validate Brazilian holiday data

HOLIDAY_PATH = (
    PROJECT_ROOT
    / CONFIG["paths"]["raw_external"]
    / "brazil_holidays.csv"
)

HOLIDAY_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save the previously acquired holiday data
holidays_csv_df.to_csv(
    HOLIDAY_PATH,
    index=False,
    encoding="utf-8"
)

# Read the saved file for validation
saved_holidays = pd.read_csv(HOLIDAY_PATH)

# Validation checks
assert len(saved_holidays) == len(holidays_df)
assert set(saved_holidays["source_year"]) == {2016, 2017, 2018}
assert saved_holidays["countryCode"].eq("BR").all()
assert HOLIDAY_PATH.is_file()

print("COMMERCEIQ - BRAZILIAN HOLIDAY DATA")

display(saved_holidays[
    ["date", "localName", "name", "global", "types"]
].head(10))

print("\nHoliday records by year:")
print(saved_holidays.groupby("source_year").size())

print(f"\nTotal records     : {len(saved_holidays)}")
print(f"Total columns     : {saved_holidays.shape[1]}")
print(f"Duplicate records : {saved_holidays.duplicated().sum()}")
print(f"Saved to          : {HOLIDAY_PATH.relative_to(PROJECT_ROOT)}")
print(f"File exists       : {HOLIDAY_PATH.is_file()}")

print("\nHoliday data saved and verified successfully.")

COMMERCEIQ - BRAZILIAN HOLIDAY DATA


,date,localName,name,global,types
0,2016-01-01,Confraternização Universal,New Year's Day,True,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,True,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,True,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,True,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,True,"[""Public""]"
5,2016-04-21,Dia de Tiradentes,Tiradentes,True,"[""Public""]"
6,2016-05-01,Dia do Trabalhador,Labour Day,True,"[""Public""]"
7,2016-05-26,Corpus Christi,Corpus Christi,True,"[""Public""]"
8,2016-07-09,Revolução Constitucionalista de 1932,Constitutionalist Revolution of 1932,False,"[""Public""]"
9,2016-09-07,Dia da Independência,Independence Day,True,"[""Public""]"



Holiday records by year:
source_year
2016    14
2017    14
2018    14
dtype: int64

Total records     : 42
Total columns     : 11
Duplicate records : 0
Saved to          : data\raw\external\brazil_holidays.csv
File exists       : True

Holiday data saved and verified successfully.


In [7]:
# Step 2.9(c): Register the external holiday dataset

import hashlib
import pandas as pd
from datetime import datetime, timezone

MANIFEST_PATH = (
    PROJECT_ROOT
    / CONFIG["paths"]["metadata"]
    / "dataset_manifest.csv"
)

manifest_df = pd.read_csv(MANIFEST_PATH)

# Calculate the holiday dataset checksum
sha256 = hashlib.sha256()

with HOLIDAY_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

holiday_stats = HOLIDAY_PATH.stat()

# Create the external dataset metadata record
holiday_metadata = {
    "dataset": "brazil_holidays",
    "source": "external",
    "filename": HOLIDAY_PATH.name,
    "relative_path": HOLIDAY_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "size_bytes": holiday_stats.st_size,
    "size_mb": round(holiday_stats.st_size / (1024 ** 2), 4),
    "last_modified_utc": datetime.fromtimestamp(
        holiday_stats.st_mtime,
        tz=timezone.utc
    ).isoformat(),
    "sha256": sha256.hexdigest(),
    "file_exists": HOLIDAY_PATH.is_file(),
    "correct_extension": HOLIDAY_PATH.suffix.lower() == ".csv",
    "non_empty": holiday_stats.st_size > 0,
    "status": "PASSED",
    "source_url": "https://date.nager.at/api/v3/PublicHolidays/{year}/BR",
    "registered_at_utc": datetime.now(timezone.utc).isoformat()
}

# Prevent duplicate registration when rerunning the cell
manifest_df = manifest_df[
    manifest_df["dataset"] != "brazil_holidays"
].copy()

manifest_df = pd.concat(
    [manifest_df, pd.DataFrame([holiday_metadata])],
    ignore_index=True
)

# Validate before saving
assert len(manifest_df) == 12
assert manifest_df["dataset"].is_unique
assert manifest_df["status"].eq("PASSED").all()
assert manifest_df["sha256"].str.fullmatch(r"[0-9a-f]{64}").all()

# Save the updated manifest
manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8"
)

# Verify the persisted manifest
saved_manifest = pd.read_csv(MANIFEST_PATH)

assert len(saved_manifest) == 12
assert saved_manifest["dataset"].is_unique
assert saved_manifest["sha256"].equals(manifest_df["sha256"])

print("COMMERCEIQ - UPDATED DATASET MANIFEST")

display(saved_manifest[
    ["dataset", "source", "size_mb", "status"]
])

print("\nDatasets by source:")
print(saved_manifest.groupby("source").size())

print(f"\nTotal registered datasets : {len(saved_manifest)}")
print(f"Manifest columns          : {saved_manifest.shape[1]}")
print(f"Duplicate dataset names   : {saved_manifest['dataset'].duplicated().sum()}")
print(f"Manifest location        : {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")

print("\nExternal dataset registered successfully.")

COMMERCEIQ - UPDATED DATASET MANIFEST


,dataset,source,size_mb,status
0,orders,ecommerce,16.8370,PASSED
1,customers,ecommerce,8.6155,PASSED
2,order_items,ecommerce,14.7235,PASSED
3,payments,ecommerce,5.5095,PASSED
4,reviews,ecommerce,13.7822,PASSED
5,products,ecommerce,2.2692,PASSED
6,sellers,ecommerce,0.1666,PASSED
7,geolocation,ecommerce,58.4353,PASSED
8,category_translation,ecommerce,0.0025,PASSED
9,marketing_leads,marketing,0.6712,PASSED



Datasets by source:
source
ecommerce    9
external     1
marketing    2
dtype: int64

Total registered datasets : 12
Manifest columns          : 14
Duplicate dataset names   : 0
Manifest location        : data\metadata\dataset_manifest.csv

External dataset registered successfully.


In [8]:
# Step 2.10: Configure ingestion logging

import logging
import sys

# Define log file location
LOG_PATH = (
    PROJECT_ROOT
    / CONFIG["paths"]["logs"]
    / "ingestion.log"
)

LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Create a dedicated ingestion logger
logger = logging.getLogger("commerceiq.ingestion")

logger.setLevel(CONFIG["logging"]["level"])
logger.propagate = False

# Prevent duplicate handlers when rerunning the cell
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
    handler.close()

# Define log message format
log_formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# File handler: persist logs to disk
file_handler = logging.FileHandler(
    LOG_PATH,
    mode="a",
    encoding="utf-8"
)

file_handler.setFormatter(log_formatter)

# Console handler: display logs in notebook
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(log_formatter)

# Attach handlers
logger.addHandler(file_handler)
logger.addHandler(console_handler)

# Record actual pipeline status
logger.info("CommerceIQ ingestion logger initialized.")
logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Registered datasets: %d", len(saved_manifest))

for row in saved_manifest.itertuples(index=False):
    logger.info(
        "Dataset: %s | Source: %s | Status: %s",
        row.dataset,
        row.source,
        row.status
    )

logger.info("Ingestion logging configuration completed.")

# Verify persisted log file
assert LOG_PATH.is_file()
assert LOG_PATH.stat().st_size > 0
assert len(logger.handlers) == 2

print("\nLOGGING VALIDATION")
print("Log file:", LOG_PATH.relative_to(PROJECT_ROOT))
print("Log file exists:", LOG_PATH.is_file())
print("Log file size (bytes):", LOG_PATH.stat().st_size)
print("Logging level:", logging.getLevelName(logger.level))
print("Active handlers:", len(logger.handlers))
print("Logging configured successfully.")

2026-11-27 12:03:54 | INFO | commerceiq.ingestion | CommerceIQ ingestion logger initialized.
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Project root: c:\Users\bgunj\Desktop\commerceIQ
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Registered datasets: 12
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: orders | Source: ecommerce | Status: PASSED
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: customers | Source: ecommerce | Status: PASSED
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: order_items | Source: ecommerce | Status: PASSED
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: payments | Source: ecommerce | Status: PASSED
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: reviews | Source: ecommerce | Status: PASSED
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: products | Source: ecommerce | Status: PASSED
2026-11-27 12:03:54 | INFO | commerceiq.ingestion | Dataset: sellers | Source: ecommerce

In [9]:
# Step 2.11: Implement exception handling and safe ingestion

import pandas as pd
import hashlib
from time import perf_counter


class DataIngestionError(Exception):
    """Custom exception for data ingestion failures."""
    pass


def calculate_file_hash(file_path, chunk_size=1024 * 1024):
    """Calculate the SHA-256 checksum of a file."""

    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


def safe_load_csv(dataset_name, file_path, expected_hash=None):
    """Validate and load a CSV dataset with error handling."""

    try:
        logger.info("Starting ingestion: %s", dataset_name)

        # Validate file existence
        if not file_path.is_file():
            raise FileNotFoundError(
                f"Source file not found: {file_path}"
            )

        # Validate file extension
        if file_path.suffix.lower() != ".csv":
            raise ValueError("Invalid file extension.")

        # Validate file size
        if file_path.stat().st_size == 0:
            raise ValueError("Source file is empty.")

        # Verify source integrity
        if expected_hash is not None:
            actual_hash = calculate_file_hash(file_path)

            if actual_hash != expected_hash:
                raise ValueError("SHA-256 checksum mismatch.")

        start_time = perf_counter()

        # Load raw data without premature type conversion
        df = pd.read_csv(
            file_path,
            encoding="utf-8",
            dtype=str,
            keep_default_na=False,
            low_memory=False,
            on_bad_lines="error"
        )

        if df.empty:
            raise ValueError("Dataset contains no records.")

        elapsed_time = perf_counter() - start_time

        logger.info(
            "Loaded: %s | Rows: %d | Columns: %d | Time: %.4f sec",
            dataset_name,
            df.shape[0],
            df.shape[1],
            elapsed_time
        )

        return df

    except (
        FileNotFoundError,
        PermissionError,
        pd.errors.EmptyDataError,
        pd.errors.ParserError,
        UnicodeDecodeError,
        ValueError,
        OSError
    ) as error:

        logger.error(
            "Ingestion failed: %s | %s",
            dataset_name,
            error
        )

        raise DataIngestionError(
            f"Unable to ingest {dataset_name}: {error}"
        ) from error


# Reload the persisted manifest
manifest_df = pd.read_csv(MANIFEST_PATH)

# Initialize ingestion containers
datasets = {}
ingestion_results = []

pipeline_start = perf_counter()

# Safely ingest all registered datasets
for row in manifest_df.itertuples(index=False):

    file_path = PROJECT_ROOT / row.relative_path

    try:
        df = safe_load_csv(
            dataset_name=row.dataset,
            file_path=file_path,
            expected_hash=row.sha256
        )

        datasets[row.dataset] = df

        ingestion_results.append({
            "dataset": row.dataset,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "status": "LOADED",
            "error": None
        })

    except DataIngestionError as error:

        ingestion_results.append({
            "dataset": row.dataset,
            "rows": 0,
            "columns": 0,
            "status": "FAILED",
            "error": str(error)
        })


ingestion_summary_df = pd.DataFrame(ingestion_results)

print("\nCOMMERCEIQ - SAFE INGESTION SUMMARY")

display(ingestion_summary_df)

successful = ingestion_summary_df["status"].eq("LOADED").sum()
failed = ingestion_summary_df["status"].eq("FAILED").sum()

print(f"Total datasets : {len(ingestion_summary_df)}")
print(f"Successful     : {successful}")
print(f"Failed         : {failed}")
print(f"Total records  : {ingestion_summary_df['rows'].sum():,}")
print(f"Execution time : {perf_counter() - pipeline_start:.2f} seconds")

if failed > 0:
    logger.error("Pipeline completed with ingestion failures.")
    raise DataIngestionError(f"{failed} datasets failed ingestion.")

logger.info("All registered datasets passed safe ingestion.")

print("Safe ingestion completed successfully.")

2026-11-27 12:04:29 | INFO | commerceiq.ingestion | Starting ingestion: orders
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Loaded: orders | Rows: 99441 | Columns: 8 | Time: 0.3365 sec
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Starting ingestion: customers
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Loaded: customers | Rows: 99441 | Columns: 5 | Time: 0.1040 sec
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Starting ingestion: order_items
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Loaded: order_items | Rows: 112650 | Columns: 7 | Time: 0.1424 sec
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Starting ingestion: payments
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Loaded: payments | Rows: 103886 | Columns: 5 | Time: 0.0679 sec
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Starting ingestion: reviews
2026-11-27 12:04:30 | INFO | commerceiq.ingestion | Loaded: reviews | Rows: 99224 | Columns: 7 | Time: 0.1686 sec
2026-11-27 12:04

,dataset,rows,columns,status,error
0,orders,99441,8,LOADED,None
1,customers,99441,5,LOADED,None
2,order_items,112650,7,LOADED,None
3,payments,103886,5,LOADED,None
4,reviews,99224,7,LOADED,None
5,products,32951,9,LOADED,None
6,sellers,3095,4,LOADED,None
7,geolocation,1000163,5,LOADED,None
8,category_translation,71,2,LOADED,None
9,marketing_leads,8000,4,LOADED,None


Total datasets : 12
Successful     : 12
Failed         : 0
Total records  : 1,559,806
Execution time : 2.10 seconds
2026-11-27 12:04:31 | INFO | commerceiq.ingestion | All registered datasets passed safe ingestion.
Safe ingestion completed successfully.


In [10]:
# Step 2.15(a): Final Phase 2 validation

import hashlib
import pandas as pd

from pathlib import Path

# Locate project root independently
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        directory
        for directory in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (directory / "config.yaml").is_file()
        and (directory / "data" / "raw").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found.")

# Required Phase 2 deliverables
required_files = [
    "notebooks/00_data_ingestion.ipynb",
    "data/raw/external/brazil_holidays.csv",
    "data/metadata/dataset_manifest.csv",
    "data/metadata/ingestion_summary.csv",
    "logs/ingestion.log",
    "src/ingestion/extract.py",
    "tests/test_ingestion.py",
    "docs/data_sources.md"
]

deliverable_results = []

for relative_path in required_files:

    file_path = PROJECT_ROOT / relative_path

    exists = file_path.is_file()
    size = file_path.stat().st_size if exists else 0

    deliverable_results.append({
        "deliverable": relative_path,
        "exists": exists,
        "size_bytes": size,
        "status": "PASSED" if exists and size > 0 else "FAILED"
    })

deliverable_validation_df = pd.DataFrame(deliverable_results)

# Load persisted metadata
manifest = pd.read_csv(
    PROJECT_ROOT / "data/metadata/dataset_manifest.csv"
)

summary = pd.read_csv(
    PROJECT_ROOT / "data/metadata/ingestion_summary.csv"
)

# Verify source file integrity again
integrity_results = []

for row in manifest.itertuples(index=False):

    file_path = PROJECT_ROOT / row.relative_path

    actual_hash = None

    if file_path.is_file():

        sha256 = hashlib.sha256()

        with file_path.open("rb") as file:

            for chunk in iter(
                lambda: file.read(1024 * 1024),
                b""
            ):
                sha256.update(chunk)

        actual_hash = sha256.hexdigest()

    integrity_results.append({
        "dataset": row.dataset,
        "file_exists": file_path.is_file(),
        "checksum_match": actual_hash == row.sha256,
        "status": (
            "PASSED"
            if actual_hash == row.sha256
            else "FAILED"
        )
    })

integrity_validation_df = pd.DataFrame(integrity_results)

# Expected dataset dimensions
expected_shapes = {
    "orders": (99441, 8),
    "customers": (99441, 5),
    "order_items": (112650, 7),
    "payments": (103886, 5),
    "reviews": (99224, 7),
    "products": (32951, 9),
    "sellers": (3095, 4),
    "geolocation": (1000163, 5),
    "category_translation": (71, 2),
    "marketing_leads": (8000, 4),
    "closed_deals": (842, 14),
    "brazil_holidays": (42, 11)
}

actual_shapes = {
    row.dataset: (row.rows, row.columns)
    for row in summary.itertuples(index=False)
}

# Consolidated validation checks
final_checks = {
    "All required deliverables available":
        deliverable_validation_df["status"].eq("PASSED").all(),

    "Exactly 12 registered datasets":
        len(manifest) == 12,

    "Unique dataset registrations":
        manifest["dataset"].is_unique,

    "Correct dataset inventory":
        set(manifest["dataset"]) == set(expected_shapes),

    "All manifest validations passed":
        manifest["status"].eq("PASSED").all(),

    "All source checksums matched":
        integrity_validation_df["checksum_match"].all(),

    "Correct source distribution":
        manifest["source"].value_counts().to_dict() == {
            "ecommerce": 9,
            "marketing": 2,
            "external": 1
        },

    "Exactly 12 ingestion results":
        len(summary) == 12,

    "Unique ingestion results":
        summary["dataset"].is_unique,

    "All datasets successfully ingested":
        summary["status"].eq("LOADED").all(),

    "Expected dataset dimensions":
        actual_shapes == expected_shapes,

    "Expected total source records":
        summary["rows"].sum() == 1559806
}

final_validation_df = pd.DataFrame([
    {
        "validation_check": check,
        "status": "PASSED" if result else "FAILED"
    }
    for check, result in final_checks.items()
])

# Display results
print("COMMERCEIQ - PHASE 2 DELIVERABLE VALIDATION\n")
display(deliverable_validation_df)

print("\nSOURCE FILE INTEGRITY VALIDATION\n")
display(integrity_validation_df)

print("\nFINAL PHASE 2 VALIDATION\n")
display(final_validation_df)

passed = final_validation_df["status"].eq("PASSED").sum()
failed = final_validation_df["status"].eq("FAILED").sum()

print(f"\nRequired deliverables : {len(required_files)}")
print(f"Registered datasets   : {len(manifest)}")
print(f"Total source records  : {summary['rows'].sum():,}")
print(f"Validation checks     : {len(final_validation_df)}")
print(f"Passed                : {passed}")
print(f"Failed                : {failed}")

assert failed == 0, "Phase 2 validation failed."

print("\nAll Phase 2 technical validation checks passed.")

COMMERCEIQ - PHASE 2 DELIVERABLE VALIDATION



,deliverable,exists,size_bytes,status
0,notebooks/00_data_ingestion.ipynb,True,84351,PASSED
1,data/raw/external/brazil_holidays.csv,True,5900,PASSED
2,data/metadata/dataset_manifest.csv,True,4120,PASSED
3,data/metadata/ingestion_summary.csv,True,1087,PASSED
4,logs/ingestion.log,True,6976,PASSED
5,src/ingestion/extract.py,True,10460,PASSED
6,tests/test_ingestion.py,True,5416,PASSED
7,docs/data_sources.md,True,14503,PASSED



SOURCE FILE INTEGRITY VALIDATION



,dataset,file_exists,checksum_match,status
0,orders,True,True,PASSED
1,customers,True,True,PASSED
2,order_items,True,True,PASSED
3,payments,True,True,PASSED
4,reviews,True,True,PASSED
5,products,True,True,PASSED
6,sellers,True,True,PASSED
7,geolocation,True,True,PASSED
8,category_translation,True,True,PASSED
9,marketing_leads,True,True,PASSED



FINAL PHASE 2 VALIDATION



,validation_check,status
0,All required deliverables available,PASSED
1,Exactly 12 registered datasets,PASSED
2,Unique dataset registrations,PASSED
3,Correct dataset inventory,PASSED
4,All manifest validations passed,PASSED
5,All source checksums matched,PASSED
6,Correct source distribution,PASSED
7,Exactly 12 ingestion results,PASSED
8,Unique ingestion results,PASSED
9,All datasets successfully ingested,PASSED



Required deliverables : 8
Registered datasets   : 12
Total source records  : 1,559,806
Validation checks     : 12
Passed                : 12
Failed                : 0

All Phase 2 technical validation checks passed.
